# AI for Handwriting Identification — Colab quickstart

**Colab has no `C:\Users\...`.** On your PC the release may live at `C:\Users\thisb\Downloads\AnyScriptFiltered.tar.gz`. For Colab: **upload that `.tar.gz` to Google Drive** (e.g. `MyDrive/`), then use a **`/content/drive/MyDrive/...`** path below and extract there so training survives disconnects.

**Dataset:** `DATA_ROOT` = folder whose **children are author IDs** (default `/content/drive/MyDrive/AnyScriptFiltered`). Layouts: `author/book/*.jpg` or `author/*.png`.

## Part A, B, C (run via `/content/ai-hw/scripts/...`; shell stays in `/content` so `pip` never breaks after `rm -rf`)

| Part | Script | Output |
|------|--------|--------|
| **A — Train** | `train_triplet_unsloth.py` | `OUT/best.pt` |
| **B — FAISS** | `build_faiss_index.py` | `OUT/faiss.index`, `OUT/meta.npy` |
| **C — Submit CSV** | `export_anyscript_submission.py` | dense `intra_book` / `extra_book` CSV |

More detail: **`README.md`** (Quick start + ICDAR submission).

In [3]:
# Optional: persist data & checkpoints on Drive
# If mount hangs, this forces a clean remount.
from google.colab import drive

try:
    drive.flush_and_unmount()
except Exception:
    pass

drive.mount("/content/drive", force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


### Reset working directory (run if you see `getcwd` / `Unable to read current working directory`)

If your shell was inside `/content/ai-hw` and that folder was deleted, **run the next cell** before **Clone** or **`git pull`**.

In [2]:
# Clone repo (fork/URL ok — change if you renamed the repo)
# Do NOT add %cd /content/ai-hw/scripts — re-run + rm deletes cwd (getcwd / git clone fail).
%cd /content
import os
os.chdir("/content")
!rm -rf /content/ai-hw
!git clone https://github.com/Todd838/AI-for-Handwriting-Identification.git /content/ai-hw
# Do not %cd into the repo: if you re-run this cell, rm -rf removes your cwd and pip breaks.
print("Repo:", "/content/ai-hw/scripts — run scripts with full path below.")

/content
Cloning into '/content/ai-hw'...
remote: Enumerating objects: 234, done.
remote: Counting objects: 100% (234/234), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 234 (delta 165), reused 133 (delta 64), pack-reused 0 (from 0)
Receiving objects: 100% (234/234), 78.98 KiB | 2.63 MiB/s, done.
Resolving deltas: 100% (165/165), done.
Repo: /content/ai-hw/scripts — run scripts with full path below.


### Update code from GitHub (optional)

If **`/content/ai-hw` already exists** from an earlier run and you only want the latest `main` branch, run the **next cell** instead of re-running **Clone**. **Skip** if you just ran Clone. If you see `fatal: not a git repository`, run **Clone** first.

In [3]:
%cd /content
!cd /content/ai-hw && git pull origin main

/content
From https://github.com/Todd838/AI-for-Handwriting-Identification
 * branch            main       -> FETCH_HEAD
Already up to date.


In [1]:
# Install everything from the repo (same as local requirements.txt)
# Safe on fresh runtimes: auto-clones repo if /content/ai-hw is missing.
import os
import subprocess

os.chdir("/content")
repo_dir = "/content/ai-hw"
req_path = f"{repo_dir}/requirements.txt"
if not os.path.isfile(req_path):
    print("/content/ai-hw missing; cloning fresh repo...")
    subprocess.run(["rm", "-rf", repo_dir], check=False)
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/Todd838/AI-for-Handwriting-Identification.git",
            repo_dir,
        ],
        check=True,
    )

%pip install -q -r /content/ai-hw/requirements.txt
# If pip errors on a package, run: %pip install -q unsloth  (after GPU runtime is on)

/content/ai-hw missing; cloning fresh repo...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 133.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 134.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.6 MB/s eta 0:00:0

In [2]:
# Unsloth (use Runtime → GPU first). If install fails, train with --no_unsloth
import os
os.chdir("/content")
import torch
print("CUDA available:", torch.cuda.is_available())
%pip install -q unsloth

CUDA available: True


### GPU required for Unsloth

**`CUDA available: False`** means this runtime has **no GPU**. Use **Runtime → Change runtime type → Hardware accelerator → T4 GPU** (or L4 / A100 if offered), then **Runtime → Restart runtime** and **re-run** cells from the top (mount → clone → install → this cell).

Without a GPU you can still try training with **`--no_unsloth`** (much slower / may OOM on large models).

In [4]:
# Dataset on Colab = Drive path, NOT C:\Users\thisb\Downloads\...
# Logic is in /content/ai-hw/scripts/colab_dataset_setup.py so `git pull` fixes imports
# even when this Colab tab still shows an old copy of the notebook.
import os
import sys

os.chdir("/content")
_setup = "/content/ai-hw/scripts/colab_dataset_setup.py"
if not os.path.isfile(_setup):
    raise FileNotFoundError(
        _setup
        + " missing. Run Clone, then: !cd /content/ai-hw && git pull origin main"
    )
_g = globals()
exec(compile(open(_setup, encoding="utf-8").read(), _setup, "exec"), _g, _g)

KeyboardInterrupt: 

### Check paths on Google Drive (optional)

Run the **next cell** after the dataset cell to see **where things are**: `DATA_ROOT`, optional **`query`** sibling, default **`OUT`**, and a short search for **`binarized`** folders. **Mount Drive** first.

In [5]:
# Where are training data, query folder, and checkpoints? (Run after dataset cell.)
import os

_DRIVE = "/content/drive/MyDrive"


def _ok(p):
    return os.path.isdir(p) or os.path.isfile(p)


def _brief_list(path, max_names=14):
    if not os.path.isdir(path):
        return "(missing)"
    names = sorted(os.listdir(path))
    tail = f" … (+{len(names) - max_names} more)" if len(names) > max_names else ""
    return ", ".join(names[:max_names]) + tail


print("Drive:", _ok("/content/drive"), "| MyDrive:", _ok(_DRIVE), "| cwd:", os.getcwd())

fixed = [
    f"{_DRIVE}/AnyScriptFiltered",
    f"{_DRIVE}/AnyScriptFiltered/AnyScriptFiltered.tar.gz",
    f"{_DRIVE}/data/datasets/AnyScriptFiltered",
    f"{_DRIVE}/data/datasets/AnyScriptFiltered/binarized",
    f"{_DRIVE}/data/datasets/AnyScriptFiltered/query",
    f"{_DRIVE}/anyscript_runs/run1",
]
print("\n--- Common paths ---")
for p in fixed:
    print(_ok(p), p)

if "DATA_ROOT" in globals() and DATA_ROOT:
    dr = os.path.abspath(DATA_ROOT.rstrip("/"))
    parent = os.path.dirname(dr)
    print("\n--- Dataset cell (DATA_ROOT) ---")
    print("DATA_ROOT:", _ok(dr), dr)
    print("  first entries:", _brief_list(dr))
    for name in ("query", "queries", "query_pages"):
        sib = os.path.join(parent, name)
        print(f"  sibling {name}/:", _ok(sib), sib)
else:
    print("\n(DATA_ROOT not set — run the dataset cell above first.)")

if "OUT" in globals() and OUT:
    print("\n--- Part A OUT ---")
    print("OUT:", _ok(OUT), OUT)
    for fn in ("best.pt", "faiss.index", "meta.npy"):
        print(f"  {fn}:", _ok(os.path.join(OUT, fn)))
else:
    print("\n(OUT not set — set OUT in the Part A cell when you train.)")

# Bounded search for folders named binarized under MyDrive/data (avoids scanning all of Drive)
seed = os.path.join(_DRIVE, "data")
print("\n--- Folders named binarized under .../MyDrive/data (max 20 hits) ---")
hits = []
if os.path.isdir(seed):
    for root, dirs, _files in os.walk(seed):
        depth = root[len(seed):].count(os.sep)
        if depth > 10:
            dirs[:] = []
            continue
        if os.path.basename(root) == "binarized":
            hits.append(root)
            dirs[:] = []
        if len(hits) >= 20:
            break
    for h in hits:
        print(" ", h)
else:
    print("  (no .../MyDrive/data — extract or upload under data/datasets if needed)")


Drive: True | MyDrive: True | cwd: /content

--- Common paths ---
True /content/drive/MyDrive/AnyScriptFiltered
True /content/drive/MyDrive/AnyScriptFiltered/AnyScriptFiltered.tar.gz
True /content/drive/MyDrive/data/datasets/AnyScriptFiltered
True /content/drive/MyDrive/data/datasets/AnyScriptFiltered/binarized
True /content/drive/MyDrive/data/datasets/AnyScriptFiltered/query
True /content/drive/MyDrive/anyscript_runs/run1

(DATA_ROOT not set — run the dataset cell above first.)

(OUT not set — set OUT in the Part A cell when you train.)

--- Folders named binarized under .../MyDrive/data (max 20 hits) ---
  /content/drive/MyDrive/data/datasets/AnyScriptFiltered/binarized


## Part A — training

Re-run the **dataset** cell above **in this session** before training so `DATA_ROOT` is set (it must not stay the empty `AnyScriptFiltered` folder). Then run the next cell. Checkpoints go to **`OUT`** on Drive.

### If Part A says `0 pages found`

1. If `AnyScriptFiltered/` only has **`.tar.gz`**, you must **extract** first (the dataset cell now tries both `MyDrive/AnyScriptFiltered.tar.gz` and `MyDrive/AnyScriptFiltered/AnyScriptFiltered.tar.gz`).
2. Use **`--data_root auto`** in the train command so path resolution runs inside `train_triplet_unsloth.py`.
3. **Do not** use `DATA_ROOT` from unrelated projects (e.g. other Kaggle folders). The inspector only suggests **AnyScript-related** paths unless you pass `--force-unrelated-scan`.

### Shell `!python` vs `{OUT}` / `{DATA_ROOT}`

Lines starting with **`!`** are run by the **shell**, not Python: **`{OUT}` and `{DATA_ROOT}` are not expanded** (you get a literal path like `{OUT}/best.pt` and `FileNotFoundError`). Use the **Part B** cell below (`subprocess` + real paths), or in `!python` use **`--data_root auto`** (same as training) and paste the full checkpoint path.

In [ ]:
import os
import subprocess
import sys

repo_dir = "/content/ai-hw"
script_path = f"{repo_dir}/scripts/inspect_anyscript_layout.py"

# Finds which subfolder actually contains page images.
# If /content/ai-hw is missing in this runtime, auto-clone it first.
# Run the Drive mount cell first — if every path shows exists=False, Drive is not mounted.
%cd /content

if not os.path.isfile(script_path):
    print("/content/ai-hw missing; cloning fresh repo...")
    subprocess.run(["rm", "-rf", repo_dir], check=False)
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/Todd838/AI-for-Handwriting-Identification.git",
            repo_dir,
        ],
        check=True,
    )

try:
    # Running without --data_root auto, as exit code 2 suggests an invalid argument
    result = subprocess.run([sys.executable, script_path], capture_output=True, text=True, check=True)
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print(f"Command failed with exit status {e.returncode}.")
    print("\n--- Standard Error Output ---")
    print(e.stderr)
    print("--- Standard Output ---")
    print(e.stdout)

In [ ]:
OUT = "/content/drive/MyDrive/anyscript_runs/run1"
import os
import subprocess
import sys

os.environ["ANYSCRIPT_OUT"] = OUT
os.makedirs(OUT, exist_ok=True)
os.chdir("/content")

base = [
    sys.executable,
    "/content/ai-hw/scripts/train_triplet_unsloth.py",
    "--data_root",
    "auto",
    "--model_name",
    "zai-org/GLM-OCR",
    "--output_dir",
    OUT,
    "--epochs",
    "3",
    "--steps_per_epoch",
    "2000",
]

attempts = [
    ("batch_size=4 (preferred)", ["--batch_size", "4"]),
    ("batch_size=2 (VRAM fallback)", ["--batch_size", "2"]),
    ("batch_size=2 + --no_unsloth (compat fallback)", ["--batch_size", "2", "--no_unsloth"]),
]

last_err = None
for label, extra in attempts:
    cmd = base + extra
    print(f"\n=== Training attempt: {label} ===")
    try:
        subprocess.run(cmd, check=True)
        print("Training completed successfully.")
        last_err = None
        break
    except subprocess.CalledProcessError as e:
        last_err = e
        print(f"Attempt failed with exit code {e.returncode}. Trying next fallback...")

if last_err is not None:
    raise RuntimeError(
        "All training attempts failed. Scroll up to the first real error message from "
        "train_triplet_unsloth.py and paste it here for a targeted fix."
    )

## Part B — FAISS index

Run after **`best.pt`** exists under `OUT`, then run the **next cell** (it uses `subprocess` + `--data_root auto`, not shell `!python` with `{OUT}` — those braces are not expanded in `!` lines).

The command enables `--resume` + periodic checkpoints, so reconnecting Colab and re-running Part B continues from saved progress.

If your notebook still shows an old commented `!python ... {DATA_ROOT} ...` Part B, open **`/content/ai-hw/colab_quickstart.ipynb`** after **Clone / git pull**, or re-open the notebook from GitHub.

In [ ]:
# Part B — full-gallery FAISS with resume checkpoints.
# Safe to re-run after disconnect: it resumes from existing faiss.index/meta.npy.
# batch_size=1 processes one page at a time to reduce memory pressure and crashes.
import os
import subprocess
import sys

if "OUT" not in globals() or not OUT:
    raise RuntimeError("Set OUT in the Part A cell above (same path as --output_dir).")
_ckpt = os.path.join(OUT, "best.pt")
if not os.path.isfile(_ckpt):
    raise FileNotFoundError(_ckpt + " — train Part A first or fix OUT.")

subprocess.run(
    [
        sys.executable,
        "/content/ai-hw/scripts/build_faiss_index.py",
        "--data_root",
        "auto",
        "--checkpoint",
        _ckpt,
        "--index_out",
        os.path.join(OUT, "faiss.index"),
        "--meta_out",
        os.path.join(OUT, "meta.npy"),
        "--batch_size",
        "1",
        "--all_pages",
        "--resume",
        "--save_every_batches",
        "25",
    ],
    check=True,
)

In [6]:
DATA_ROOT = "/content/drive/MyDrive/data/datasets/AnyScriptFiltered/binarized"
OUT = "/content/drive/MyDrive/anyscript_runs/run1"
GALLERY_ROOT = DATA_ROOT
QUERY_ROOT = "/content/drive/MyDrive/data/datasets/AnyScriptFiltered/query"

import os
print("DATA_ROOT:", os.path.isdir(DATA_ROOT), DATA_ROOT)
print("OUT:", os.path.isdir(OUT), OUT)
print("best.pt:", os.path.isfile(os.path.join(OUT, "best.pt")))
print("GALLERY_ROOT:", os.path.isdir(GALLERY_ROOT), GALLERY_ROOT)
print("QUERY_ROOT:", os.path.isdir(QUERY_ROOT), QUERY_ROOT)

DATA_ROOT: True /content/drive/MyDrive/data/datasets/AnyScriptFiltered/binarized
OUT: True /content/drive/MyDrive/anyscript_runs/run1
best.pt: True
GALLERY_ROOT: True /content/drive/MyDrive/data/datasets/AnyScriptFiltered/binarized
QUERY_ROOT: True /content/drive/MyDrive/data/datasets/AnyScriptFiltered/query


## Part C — submission CSV

**Gallery** defaults to **`DATA_ROOT`** (training pages), e.g. `.../AnyScriptFiltered/binarized` on Drive.

**Query** must be a **separate** tree with the same `author/book/page` layout (official query release). The next cell **auto-picks** the first folder that exists among siblings of `binarized` (`query`, `queries`, …) and a few fixed Drive paths. If nothing is found, add your query images under e.g. **`.../AnyScriptFiltered/query`** or set **`QUERY_ROOT`** manually.

For a real upload, set **`QUERY_IDS_JSON`** and **`GALLERY_IDS_JSON`** too (see **`README.md`**). Use synthetic IDs only for local testing.

In [ ]:
# --- Part C (full data, no official ID maps yet) ---
GALLERY_ROOT = "/content/drive/MyDrive/data/datasets/AnyScriptFiltered/binarized"
QUERY_ROOT = "/content/drive/MyDrive/data/datasets/AnyScriptFiltered/query"
QUERY_IDS_JSON = None
GALLERY_IDS_JSON = None
ALLOW_SYNTHETIC_IDS = True

import os
import subprocess
import sys

OUT = "/content/drive/MyDrive/anyscript_runs/run1"  # set your run folder
if not os.path.isdir(OUT):
    raise RuntimeError(f"OUT folder missing: {OUT}")

if not QUERY_ROOT or not os.path.isdir(QUERY_ROOT):
    raise RuntimeError(f"Set QUERY_ROOT to an existing folder, got: {QUERY_ROOT!r}")
if not GALLERY_ROOT or not os.path.isdir(GALLERY_ROOT):
    raise RuntimeError(f"Set GALLERY_ROOT to an existing folder, got: {GALLERY_ROOT!r}")

_ckpt = os.path.join(OUT, "best.pt")
if not os.path.isfile(_ckpt):
    raise FileNotFoundError(_ckpt + " — train Part A first or fix OUT.")

os.chdir("/content")

def run_submission(mode: str, out_name: str):
    cmd = [
        sys.executable,
        "/content/ai-hw/scripts/export_anyscript_submission.py",
        "--mode", mode,
        "--checkpoint", _ckpt,
        "--gallery_data_root", GALLERY_ROOT,
        "--query_data_root", QUERY_ROOT,
        "--out_csv", os.path.join(OUT, out_name),
        "--batch_size", "1",
        "--query_chunk", "1",
    ]
    if ALLOW_SYNTHETIC_IDS:
        cmd.append("--allow_synthetic_ids")
    if QUERY_IDS_JSON:
        cmd.extend(["--query_ids_json", QUERY_IDS_JSON])
    if GALLERY_IDS_JSON:
        cmd.extend(["--gallery_ids_json", GALLERY_IDS_JSON])

    print("\nRunning:", " ".join(cmd))
    subprocess.run(cmd, check=True)

run_submission("intra_book", "submission_intra_book.csv")
run_submission("extra_book", "submission_extra_book.csv")

print("\nDone.")
print("intra:", os.path.join(OUT, "submission_intra_book.csv"))
print("extra:", os.path.join(OUT, "submission_extra_book.csv"))